# Assignment 03 - Edge Computing
## 23i-3022 Section A
**GPU T4 x2 + Internet ON**


In [1]:
!pip install onnx onnxruntime -q

import os, time, copy, csv
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.ao.quantization as quant
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", dev)
if dev.type == "cuda":
    print("GPU:", torch.cuda.get_device_name())

os.chdir("/kaggle/working")
for d in ["models", "models/onnx", "results"]:
    os.makedirs(d, exist_ok=True)


Device: cuda
GPU: Tesla T4


## Data


In [2]:
MU = (0.4914, 0.4822, 0.4465)
SD = (0.2470, 0.2435, 0.2616)
CLS = ("airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck")

tr_tf = T.Compose([T.RandomCrop(32,4), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(MU,SD)])
te_tf = T.Compose([T.ToTensor(), T.Normalize(MU,SD)])

trset = torchvision.datasets.CIFAR10("data", train=True,  download=True, transform=tr_tf)
teset = torchvision.datasets.CIFAR10("data", train=False, download=True, transform=te_tf)
calset = torchvision.datasets.CIFAR10("data", train=True,  download=False, transform=te_tf)

tr_dl = DataLoader(trset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
te_dl = DataLoader(teset, batch_size=100, shuffle=False, num_workers=2, pin_memory=True)
cal_dl = DataLoader(Subset(calset, torch.randperm(len(calset))[:1000].tolist()), batch_size=32, shuffle=False)

print("Train:", len(trset), " Test:", len(teset))


100%|██████████| 170M/170M [11:24<00:00, 249kB/s] 


Train: 50000  Test: 10000


## Model


In [3]:
class Block(nn.Module):
    def __init__(self, ci, co, s=1, uq=False):
        super().__init__()
        self.c1 = nn.Conv2d(ci, co, 3, s, 1, bias=False)
        self.b1 = nn.BatchNorm2d(co)
        self.r1 = nn.ReLU(inplace=True)
        self.c2 = nn.Conv2d(co, co, 3, 1, 1, bias=False)
        self.b2 = nn.BatchNorm2d(co)
        self.r2 = nn.ReLU(inplace=True)
        self.uq = uq
        self.sc = nn.Sequential()
        if s != 1 or ci != co:
            self.sc = nn.Sequential(nn.Conv2d(ci, co, 1, s, bias=False), nn.BatchNorm2d(co))
        if uq:
            self.fa = torch.ao.nn.quantized.FloatFunctional()

    def forward(self, x):
        h = self.r1(self.b1(self.c1(x)))
        h = self.b2(self.c2(h))
        h = self.fa.add(h, self.sc(x)) if self.uq else h + self.sc(x)
        return self.r2(h)


class Net(nn.Module):
    def __init__(self, nc=10, ch=None, uq=False):
        super().__init__()
        ch = ch or [64, 128, 256, 512]
        self.ch = list(ch)
        self.uq = uq
        if uq:
            self.qs = quant.QuantStub()
            self.ds = quant.DeQuantStub()
        self.c0 = nn.Conv2d(3, ch[0], 3, 1, 1, bias=False)
        self.b0 = nn.BatchNorm2d(ch[0])
        self.rl = nn.ReLU(inplace=True)
        self.l1 = self._layer(ch[0], ch[0], 2, 1)
        self.l2 = self._layer(ch[0], ch[1], 2, 2)
        self.l3 = self._layer(ch[1], ch[2], 2, 2)
        self.l4 = self._layer(ch[2], ch[3], 2, 2)
        self.ap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(ch[3], nc)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)

    def _layer(self, ci, co, n, s):
        return nn.Sequential(Block(ci, co, s, self.uq), *[Block(co, co, 1, self.uq) for _ in range(n-1)])

    def forward(self, x):
        if self.uq: x = self.qs(x)
        x = self.rl(self.b0(self.c0(x)))
        x = self.l4(self.l3(self.l2(self.l1(x))))
        x = self.ap(x).flatten(1)
        x = self.fc(x)
        if self.uq: x = self.ds(x)
        return x

    def fuse(self, qat=False):
        fn = getattr(quant, "fuse_modules_qat", quant.fuse_modules) if qat else quant.fuse_modules
        fn(self, ["c0","b0","rl"], inplace=True)
        for ly in [self.l1, self.l2, self.l3, self.l4]:
            for bk in ly:
                fn(bk, ["c1","b1","r1"], inplace=True)
                fn(bk, ["c2","b2"], inplace=True)
                if len(bk.sc) > 0:
                    fn(bk.sc, ["0","1"], inplace=True)

print("Model OK")


Model OK


## Helpers


In [4]:
def train_ep(m, dl, crit, opt):
    m.train()
    ls, ok, n = 0.0, 0, 0
    for x, y in dl:
        x, y = x.to(dev), y.to(dev)
        opt.zero_grad()
        o = m(x); l = crit(o, y)
        l.backward(); opt.step()
        ls += l.item()*x.size(0)
        ok += o.argmax(1).eq(y).sum().item()
        n += y.size(0)
    return ls/n, 100.*ok/n

@torch.no_grad()
def test(m, dl, d=dev):
    m.eval()
    ok, n = 0, 0
    for x, y in dl:
        x, y = x.to(d), y.to(d)
        ok += m(x).argmax(1).eq(y).sum().item()
        n += y.size(0)
    return 100.*ok/n

def npar(m): return sum(p.numel() for p in m.parameters())
def msz(p): return os.path.getsize(p)/1048576
def sparsity(m):
    z = sum((p==0).sum().item() for p in m.parameters())
    t = sum(p.numel() for p in m.parameters())
    return z/t if t else 0


---
# 1. Train Baseline


In [5]:
m = Net().to(dev)
print("Params:", npar(m))

crit = nn.CrossEntropyLoss()
opt = optim.SGD(m.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
sch = optim.lr_scheduler.CosineAnnealingLR(opt, 100)

best = 0
for ep in range(100):
    ls, tra = train_ep(m, tr_dl, crit, opt)
    tea = test(m, te_dl)
    sch.step()
    if (ep+1) % 10 == 0:
        print(f"Ep {ep+1}  loss={ls:.4f}  train={tra:.1f}  test={tea:.1f}")
    if tea > best:
        best = tea
        torch.save({"sd": m.state_dict(), "acc": tea, "ch": [64,128,256,512]},
                    "models/resnet18_fp32.pth")

print(f"Done. Best={best:.2f}%  Size={msz('models/resnet18_fp32.pth'):.1f}MB")


Params: 11173962
Ep 10  loss=0.5459  train=81.2  test=76.8
Ep 20  loss=0.3962  train=86.5  test=74.2
Ep 30  loss=0.3286  train=88.8  test=84.5
Ep 50  loss=0.2189  train=92.5  test=86.0
Ep 60  loss=0.1521  train=94.8  test=87.7
Ep 70  loss=0.0857  train=97.0  test=91.2
Ep 80  loss=0.0254  train=99.2  test=93.5
Ep 90  loss=0.0037  train=100.0  test=94.7
Ep 100  loss=0.0024  train=100.0  test=94.8
Done. Best=94.83%  Size=42.7MB


---
# 2. PTQ


In [6]:
mp = Net(uq=True)
ck = torch.load("models/resnet18_fp32.pth", map_location="cpu", weights_only=True)
mp.load_state_dict(ck["sd"], strict=False)

fp32a = test(mp, te_dl, "cpu")
print("FP32 acc:", round(fp32a, 2))

torch.backends.quantized.engine = "x86"
mp.eval(); mp.fuse(qat=False)
mp.qconfig = quant.get_default_qconfig("x86")
quant.prepare(mp, inplace=True)

with torch.no_grad():
    for x, _ in cal_dl:
        mp(x)

quant.convert(mp, inplace=True)
pa = test(mp, te_dl, "cpu")
print(f"PTQ acc: {pa:.2f}  drop: {fp32a-pa:.2f}")

torch.save({"sd": mp.state_dict(), "acc": pa, "fp32": fp32a, "be": "x86"},
           "models/resnet18_ptq_int8.pth")
print("Saved PTQ model")


FP32 acc: 94.83


/tmp/ipykernel_50/3479942044.py:11: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quant.prepare(mp, inplace=True)
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super()._

PTQ acc: 94.77  drop: 0.06
Saved PTQ model


---
# 3. QAT


In [8]:
mq = Net(uq=True)
ck = torch.load("models/resnet18_fp32.pth", map_location="cpu", weights_only=True)
mq.load_state_dict(ck["sd"], strict=False)

torch.backends.quantized.engine = "x86"
mq.train(); mq.fuse(qat=True)
mq.qconfig = quant.get_default_qat_qconfig("x86")
quant.prepare_qat(mq, inplace=True)
mq.to(dev)

crit = nn.CrossEntropyLoss()
opt = optim.SGD(mq.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)
sch = optim.lr_scheduler.CosineAnnealingLR(opt, 10)

for ep in range(10):
    if ep >= 3:
        mq.apply(quant.disable_observer)
    ls, tra = train_ep(mq, tr_dl, crit, opt)
    tmp = copy.deepcopy(mq).cpu().eval()
    quant.convert(tmp, inplace=True)
    tea = test(tmp, te_dl, "cpu")
    sch.step()
    print(f"Ep {ep+1}  loss={ls:.4f}  train={tra:.1f}  test={tea:.1f}")

mq.cpu().eval()
fq = copy.deepcopy(mq)
quant.convert(fq, inplace=True)
qa = test(fq, te_dl, "cpu")
print(f"QAT acc: {qa:.2f}  drop: {fp32a-qa:.2f}")

torch.save({"sd": fq.state_dict(), "acc": qa, "fp32": fp32a, "be": "x86"},
           "models/resnet18_qat_int8.pth")
print("Saved QAT model")
torch.cuda.empty_cache()


/tmp/ipykernel_50/3396437538.py:8: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quant.prepare_qat(mq, inplace=True)
/tmp/ipykernel_50/3396437538.py:20: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please 

Ep 1  loss=0.0032  train=100.0  test=94.7
Ep 2  loss=0.0031  train=100.0  test=94.7
Ep 3  loss=0.0029  train=100.0  test=94.7
Ep 4  loss=0.0029  train=100.0  test=94.7
Ep 5  loss=0.0029  train=100.0  test=94.7
Ep 6  loss=0.0026  train=100.0  test=94.7
Ep 7  loss=0.0027  train=100.0  test=94.7
Ep 8  loss=0.0028  train=100.0  test=94.8
Ep 9  loss=0.0028  train=100.0  test=94.8
Ep 10  loss=0.0027  train=100.0  test=94.8


/tmp/ipykernel_50/3396437538.py:27: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quant.convert(fq, inplace=True)


QAT acc: 94.80  drop: 0.03
Saved QAT model


---
# 4. Unstructured Pruning


In [11]:
mu = Net().to(dev)
ck = torch.load("models/resnet18_fp32.pth", map_location=dev, weights_only=True)
mu.load_state_dict(ck["sd"], strict=False)

oa = test(mu, te_dl)
print("Original acc:", round(oa, 2))

crit = nn.CrossEntropyLoss()
target_sp = 0.70

for rnd in range(1, 4):
    t = 1 - (1 - target_sp)**(rnd/3)
    cur = sparsity(mu)
    if cur >= t:
        continue
    amt = min((t - cur)/(1 - cur), 0.95)
    print(f"\nRound {rnd}  target={t*100:.0f}%  amount={amt*100:.0f}%")

    params = [(m, "weight") for m in mu.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
    prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=amt)
    print(f"  Sparsity: {sparsity(mu)*100:.1f}%  Acc: {test(mu,te_dl):.1f}%")

    opt = optim.SGD(mu.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, 10)
    for ep in range(10):
        ls, tra = train_ep(mu, tr_dl, crit, opt)
        tea = test(mu, te_dl)
        sch.step()
        print(f"  FT {ep+1}  loss={ls:.4f}  train={tra:.1f}  test={tea:.1f}")

for m in mu.modules():
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        try: prune.remove(m, "weight")
        except: pass

ua = test(mu, te_dl)
print(f"\nDone. Sparsity={sparsity(mu)*100:.1f}%  Acc={ua:.1f}%")

torch.save({"sd": mu.state_dict(), "acc": ua, "sp": sparsity(mu), "ch": [64,128,256,512]},
           "models/resnet18_unstructured_pruned.pth")
print("Saved unstructured model")
torch.cuda.empty_cache()


Original acc: 94.83

Round 1  target=33%  amount=33%
  Sparsity: 0.0%  Acc: 94.8%
  FT 1  loss=0.0054  train=99.9  test=94.0
  FT 2  loss=0.0185  train=99.4  test=92.6
  FT 3  loss=0.0188  train=99.4  test=93.6
  FT 4  loss=0.0153  train=99.6  test=94.0
  FT 5  loss=0.0092  train=99.8  test=94.3
  FT 6  loss=0.0056  train=99.9  test=94.4
  FT 7  loss=0.0037  train=99.9  test=94.7
  FT 8  loss=0.0028  train=100.0  test=94.8
  FT 9  loss=0.0028  train=99.9  test=94.9
  FT 10  loss=0.0027  train=100.0  test=94.9

Round 2  target=55%  amount=55%
  Sparsity: 0.0%  Acc: 93.9%
  FT 1  loss=0.0097  train=99.9  test=94.5
  FT 2  loss=0.0080  train=99.8  test=94.6
  FT 3  loss=0.0064  train=99.9  test=94.2
  FT 4  loss=0.0072  train=99.8  test=94.0
  FT 5  loss=0.0054  train=99.9  test=94.5
  FT 6  loss=0.0032  train=100.0  test=94.7
  FT 7  loss=0.0025  train=100.0  test=94.7
  FT 8  loss=0.0023  train=100.0  test=94.7
  FT 9  loss=0.0024  train=100.0  test=94.8
  FT 10  loss=0.0020  train=100.

---
# 5. Structured Pruning


In [14]:
def cp_conv(dst, src, oi, ii):
    w = src.weight.data
    if ii is not None: w = w[:, ii]
    if oi is not None: w = w[oi]
    dst.weight.data.copy_(w)

def cp_bn(dst, src, i):
    dst.weight.data.copy_(src.weight.data[i])
    dst.bias.data.copy_(src.bias.data[i])
    dst.running_mean.copy_(src.running_mean[i])
    dst.running_var.copy_(src.running_var[i])

orig = Net().to(dev)
ck = torch.load("models/resnet18_fp32.pth", map_location=dev, weights_only=True)
orig.load_state_dict(ck["sd"], strict=False)
oa = test(orig, te_dl)
print(f"Original: {npar(orig)} params  acc={oa:.1f}%")

ratio = 0.30
layers = [orig.l1, orig.l2, orig.l3, orig.l4]
oc = [64, 128, 256, 512]
ki = []
for ly, c in zip(layers, oc):
    imp = sum(bk.c2.weight.data.abs().sum(dim=(1,2,3)) for bk in ly)
    nk = max(8, int(c*(1-ratio)))
    ki.append(torch.topk(imp, nk).indices.sort().values)

nc = [len(k) for k in ki]
print("New channels:", nc)

orig.cpu()
ki = [k.cpu() for k in ki]
pr = Net(ch=nc)

cp_conv(pr.c0, orig.c0, ki[0], None)
cp_bn(pr.b0, orig.b0, ki[0])

pairs = [(pr.l1,orig.l1,ki[0],ki[0]), (pr.l2,orig.l2,ki[1],ki[0]),
         (pr.l3,orig.l3,ki[2],ki[1]), (pr.l4,orig.l4,ki[3],ki[2])]
for nl, ol, ci, pi in pairs:
    for bi in range(len(nl)):
        ii = pi if bi==0 else ci
        cp_conv(nl[bi].c1, ol[bi].c1, ci, ii)
        cp_bn(nl[bi].b1, ol[bi].b1, ci)
        cp_conv(nl[bi].c2, ol[bi].c2, ci, ci)
        cp_bn(nl[bi].b2, ol[bi].b2, ci)
        if len(ol[bi].sc) > 0 and len(nl[bi].sc) > 0:
            cp_conv(nl[bi].sc[0], ol[bi].sc[0], ci, ii)
            cp_bn(nl[bi].sc[1], ol[bi].sc[1], ci)

pr.fc.weight.data.copy_(orig.fc.weight.data[:, ki[3]])
pr.fc.bias.data.copy_(orig.fc.bias.data)
pr.to(dev)

print(f"Pruned: {npar(pr)} params  acc={test(pr,te_dl):.1f}%")

crit = nn.CrossEntropyLoss()
opt = optim.SGD(pr.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
sch = optim.lr_scheduler.CosineAnnealingLR(opt, 20)

for ep in range(20):
    ls, tra = train_ep(pr, tr_dl, crit, opt)
    tea = test(pr, te_dl)
    sch.step()
    print(f"Ep {ep+1}  loss={ls:.4f}  train={tra:.1f}  test={tea:.1f}")

sa = test(pr, te_dl)
print(f"\nDone. Acc={sa:.1f}%")

torch.save({"sd": pr.state_dict(), "acc": sa, "ch": pr.ch, "ratio": ratio},
           "models/resnet18_structured_pruned.pth")
print("Saved structured model")
torch.cuda.empty_cache()

Original: 11173962 params  acc=94.8%
New channels: [44, 89, 179, 358]
Pruned: 5459866 params  acc=35.2%
Ep 1  loss=0.2149  train=92.8  test=89.6
Ep 2  loss=0.1519  train=94.9  test=89.8
Ep 3  loss=0.1199  train=96.0  test=90.9
Ep 4  loss=0.1115  train=96.3  test=90.8
Ep 5  loss=0.0953  train=96.7  test=91.0
Ep 6  loss=0.0843  train=97.2  test=91.6
Ep 7  loss=0.0731  train=97.5  test=91.8
Ep 8  loss=0.0596  train=98.0  test=92.4
Ep 9  loss=0.0436  train=98.6  test=92.5
Ep 10  loss=0.0364  train=98.8  test=92.8
Ep 11  loss=0.0274  train=99.2  test=93.3
Ep 12  loss=0.0190  train=99.5  test=93.9
Ep 13  loss=0.0155  train=99.6  test=93.9
Ep 14  loss=0.0113  train=99.7  test=94.2
Ep 15  loss=0.0089  train=99.8  test=94.3
Ep 16  loss=0.0075  train=99.9  test=94.4
Ep 17  loss=0.0068  train=99.9  test=94.5
Ep 18  loss=0.0063  train=99.9  test=94.4
Ep 19  loss=0.0054  train=99.9  test=94.4
Ep 20  loss=0.0057  train=99.9  test=94.4

Done. Acc=94.4%
Saved structured model


---
# 6. Inference


In [16]:
MODELS = {
    "fp32":        ("models/resnet18_fp32.pth",                False),
    "ptq_int8":    ("models/resnet18_ptq_int8.pth",           True),
    "qat_int8":    ("models/resnet18_qat_int8.pth",           True),
    "unstructured":("models/resnet18_unstructured_pruned.pth", False),
    "structured":  ("models/resnet18_structured_pruned.pth",   False),
}

def load_model(name, path, isq):
    ck = torch.load(path, map_location="cpu", weights_only=True)
    ch = ck.get("ch", [64,128,256,512])
    if isq:
        be = ck.get("be", "x86")
        torch.backends.quantized.engine = be
        m = Net(ch=ch, uq=True); m.eval(); m.fuse(qat=False)
        cfg = quant.get_default_qat_qconfig(be) if "qat" in name else quant.get_default_qconfig(be)
        m.qconfig = cfg
        quant.prepare(m, inplace=True); quant.convert(m, inplace=True)
        m.load_state_dict(ck["sd"])
    else:
        m = Net(ch=ch); m.load_state_dict(ck["sd"])
    m.eval()
    return m

rows = []
for name, (path, isq) in MODELS.items():
    if not os.path.exists(path):
        print(f"SKIP {name}"); continue
    m = load_model(name, path, isq)
    preds, labels = [], []
    ok, n = 0, 0
    t0 = time.perf_counter()
    with torch.no_grad():
        for x, y in te_dl:
            p = m(x).argmax(1)
            ok += p.eq(y).sum().item(); n += y.size(0)
            preds += p.tolist(); labels += y.tolist()
    t1 = time.perf_counter() - t0
    acc = 100.*ok/n

    with open(f"results/predictions_{name}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["idx","true","pred","correct"])
        for i,(t,p) in enumerate(zip(labels,preds)):
            w.writerow([i, t, p, int(t==p)])

    rows.append({"model":name, "acc":round(acc,2), "time":round(t1,2),
                 "size_mb":round(msz(path),2), "params":npar(m)})
    print(f"{name}: acc={acc:.1f}%  time={t1:.1f}s  size={msz(path):.1f}MB")

with open("results/inference_summary.csv","w",newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
print("Saved inference results")


fp32: acc=94.8%  time=67.2s  size=42.7MB


/usr/local/lib/python3.13/dist-packages/torch/_utils.py:476: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,
/tmp/ipykernel_50/3600662882.py:18: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quant

ptq_int8: acc=94.8%  time=46.5s  size=10.8MB


/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:368: UserWarning: must run observer before calling calculate_qparams. Returning default values.
  if not check_min_max_valid(min_val, max_val):


qat_int8: acc=94.8%  time=47.1s  size=10.8MB
unstructured: acc=94.5%  time=67.6s  size=42.7MB
structured: acc=94.4%  time=44.0s  size=20.9MB
Saved inference results


---
# 7. Benchmark


In [18]:
brows = []
for name, (path, isq) in MODELS.items():
    if not os.path.exists(path): continue
    m = load_model(name, path, isq)
    acc = test(m, te_dl, "cpu")
    x1 = next(iter(te_dl))[0][:1]
    for _ in range(10): m(x1)
    ts = []
    for _ in range(100):
        t0 = time.perf_counter(); m(x1); ts.append((time.perf_counter()-t0)*1000)
    ts = np.array(ts)
    brows.append({
        "model":name, "acc":round(acc,2),
        "mean_ms":round(ts.mean(),2), "p50_ms":round(np.median(ts),2),
        "p95_ms":round(np.percentile(ts,95),2), "p99_ms":round(np.percentile(ts,99),2),
        "size_mb":round(msz(path),2), "params":npar(m), "sparsity":round(sparsity(m)*100,1),
    })
    print(f"{name}: acc={acc:.1f}  mean={ts.mean():.1f}ms  size={msz(path):.1f}MB  sp={sparsity(m)*100:.0f}%")

with open("results/benchmark_results.csv","w",newline="") as f:
    w = csv.DictWriter(f, fieldnames=brows[0].keys()); w.writeheader(); w.writerows(brows)
print("Saved benchmark")


fp32: acc=94.8  mean=14.3ms  size=42.7MB  sp=0%


/tmp/ipykernel_50/3600662882.py:18: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quant.prepare(m, inplace=True); quant.convert(m, inplace=True)


ptq_int8: acc=94.8  mean=7.0ms  size=10.8MB  sp=0%
qat_int8: acc=94.8  mean=7.1ms  size=10.8MB  sp=0%
unstructured: acc=94.5  mean=14.1ms  size=42.7MB  sp=91%
structured: acc=94.4  mean=11.1ms  size=20.9MB  sp=0%
Saved benchmark


In [22]:
!pip install onnxscript -q
import onnx
import onnxruntime as ort

---
# 8. ONNX Export


In [23]:
import onnx
import onnxruntime as ort

orows = []
for name in ["fp32", "unstructured", "structured"]:
    path = f"models/resnet18_{name}_pruned.pth" if name != "fp32" else "models/resnet18_fp32.pth"
    if not os.path.exists(path): continue
    ck = torch.load(path, map_location="cpu", weights_only=True)
    ch = ck.get("ch", [64,128,256,512])
    m = Net(ch=ch); m.load_state_dict(ck["sd"]); m.eval()

    op = f"models/onnx/resnet18_{name}.onnx"
    torch.onnx.export(m, torch.randn(1,3,32,32), op, opset_version=13,
                      input_names=["input"], output_names=["output"],
                      dynamic_axes={"input":{0:"b"},"output":{0:"b"}})
    onnx.checker.check_model(onnx.load(op))

    so = ort.SessionOptions(); so.intra_op_num_threads = 1
    se = ort.InferenceSession(op, so, providers=["CPUExecutionProvider"])
    inp = se.get_inputs()[0].name
    ok, n = 0, 0
    for x, y in te_dl:
        o = se.run(None, {inp: x.numpy()})
        ok += (np.argmax(o[0],1)==y.numpy()).sum(); n += y.size(0)
    acc = 100.*ok/n
    orows.append({"model":name, "acc":round(acc,2), "onnx_mb":round(msz(op),2)})
    print(f"{name}: onnx acc={acc:.1f}%  size={msz(op):.1f}MB")

with open("results/onnx_benchmark.csv","w",newline="") as f:
    w = csv.DictWriter(f, fieldnames=orows[0].keys()); w.writeheader(); w.writerows(orows)
print("Saved ONNX results")


/tmp/ipykernel_50/3660644328.py:13: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(m, torch.randn(1,3,32,32), op, opset_version=13,
W1113 21:08:02.795000 50 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `Net([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Net([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 13).
Failed to convert the model to the target version 13 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_ve

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/tmp/ipykernel_50/3660644328.py:13: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(m, torch.randn(1,3,32,32), op, opset_version=13,
W1113 21:09:58.219000 50 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


fp32: onnx acc=94.8%  size=0.1MB
[torch.onnx] Obtain model graph for `Net([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Net([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


... (2 outputs left out)

/tmp/ipykernel_50/3660644328.py:13: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(m, torch.randn(1,3,32,32), op, opset_version=13,
W1113 21:11:53.426000 50 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


unstructured: onnx acc=94.5%  size=0.1MB
[torch.onnx] Obtain model graph for `Net([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Net([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 13).
Failed to convert the model to the target version 13 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_ve

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
structured: onnx acc=94.4%  size=0.1MB
Saved ONNX results


---
# 9. Download


In [24]:
for d in ["models", "models/onnx", "results"]:
    if os.path.isdir(d):
        for f in sorted(os.listdir(d)):
            fp = os.path.join(d, f)
            if os.path.isfile(fp):
                print(f"{fp}  {os.path.getsize(fp)/1048576:.1f}MB")

import shutil
shutil.make_archive("/kaggle/working/assignment3_outputs", "zip", "/kaggle/working", ".")
print("\nassignment3_outputs.zip ready - download from Output tab")


models/resnet18_fp32.pth  42.7MB
models/resnet18_ptq_int8.pth  10.8MB
models/resnet18_qat_int8.pth  10.8MB
models/resnet18_structured_pruned.pth  20.9MB
models/resnet18_unstructured_pruned.pth  42.7MB
models/onnx/resnet18_fp32.onnx  0.1MB
models/onnx/resnet18_fp32.onnx.data  42.6MB
models/onnx/resnet18_structured.onnx  0.1MB
models/onnx/resnet18_structured.onnx.data  20.8MB
models/onnx/resnet18_unstructured.onnx  0.1MB
models/onnx/resnet18_unstructured.onnx.data  42.6MB
results/benchmark_results.csv  0.0MB
results/inference_summary.csv  0.0MB
results/onnx_benchmark.csv  0.0MB
results/predictions_fp32.csv  0.1MB
results/predictions_ptq_int8.csv  0.1MB
results/predictions_qat_int8.csv  0.1MB
results/predictions_structured.csv  0.1MB
results/predictions_unstructured.csv  0.1MB

assignment3_outputs.zip ready - download from Output tab


In [25]:
!cp assignment3_outputs.zip /kaggle/working/download.zip && echo "Ready"
from IPython.display import FileLink
FileLink("download.zip")

Ready


/kaggle/working/download.zip